
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 30px; border-radius: 15px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; box-shadow: 0 10px 20px rgba(0,0,0,0.19), 0 6px 6px rgba(0,0,0,0.23);">
    <div style="font-size: 50px; margin-bottom: 10px;"> 🕸️</div>
    <h1 style="margin: 0; font-size: 36px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
        Tox21: Deep Graph Network
    </h1>
    <p style="font-size: 18px; margin-top: 5px; font-weight: 300;">
        Deep Graph Neural Network for multi-task molecular prediction
    </p>
    <p style="font-size: 16px; margin-top: 10px; font-weight: 300; line-height: 1.5;">
    The notebook is structured as follows: <br>
     <strong>Imports</strong> |  <strong>Dataset</strong> |  <strong>GNN Model</strong> |  <strong>Training</strong> |  <strong>Hyperparameter Search</strong> |  <strong>Test</strong>
</p>

</div>

 

# **Imports**
---

In [ ]:
import torch
import optuna
import warnings
import pandas as pd
import numpy as np
from src.DGN.utils import set_seed
from src.DGN.config import *
set_seed(RANDOM_SEED)
import seaborn as sns
import matplotlib.pyplot as plt


from src.DGN.model import Tox21GNN
from src.DGN.dataset import Dataset_tox21
from src.DGN.hyper_search import objective
from src.DGN.inference import get_predictions
from src.DGN.train import compute_class_weights, create_lr_scheduler, train


from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
from sklearn.metrics import confusion_matrix, roc_auc_score
from optuna.visualization import plot_optimization_history, plot_param_importances

warnings.filterwarnings("ignore", message=".*torch-scatter.*")


# **Dataset** 
---
### Tox21  + extra

Tox21 dataset is a multi-label classification benchmark containing about 8000 molecules, each of them labelled with 12 toxicity tests. Each molecule is represented as a molecular graph (atom-node, bond-edge).

We inserted 2 extra toxicity-related properties which are `Molecular Weight (MW)` and `LogP(lipofily)`. Moreover, we added also another entry for `Global Toxicity` which is assigned to 1 if there is at least one test that highlights toxicity and 0 otherwise.

In [ ]:
tox21_df=pd.read_csv("datasets/tox21_processed_features.csv")
print(f"Uploaded Dataset. Dimensions: {tox21_df.shape}")


target_cols = [c for c in tox21_df.columns if c.startswith(('NR-', 'SR-'))]
smiles=tox21_df["smiles"].values
labels=tox21_df[target_cols].values.astype(float)
global_features=tox21_df[["MW","LogP"]].values.astype(float)


print(f"- {len(smiles)} Entries")
print(f"- Labels Shape: {labels.shape}")
print(f"- Global Feature Shape: {global_features.shape}")
#tox21_df.head()

### Dataloader 
Here we transform molecules from string (SMILES) into graphs. In particular, the `Data` object returned by the convertion_to_graph into `Dataset_tox21` class is a graph where, for each molecule, `x` are the atoms and related properties, `edge_index` are the chemical bonds, `edge_attr` are bonds properties. Going deeper, functions that extract them are:

- `get_atom_features` which collects the following atom's features:
    - *Atomic symbol*
    - *Degree of connectivity*
    - *Number of attached hydrogen atoms*
    - *Scaled atomic number*
    - *Formal charge*
    - *Hybridization state*
    - *Aromaticity*
    - *H-bond donor*
    - *H-bond acceptor*
    - *chirality*

- `get_bond_features` which collects the following bond's features:
    - *Bond Type*
    - *Ring*
    - *Conjugated*
    - *Stereochemistry*


In addition to these, the class accepts global parameters (which don't belong to the graph structure directly but that are meaningful) which are then normalized and added to the Data object.


The split is done as follow:
- |**Train**| = 70% of dataset 
- |**Validation**| = 15% of dataset 
- |**Test**| = 15% of dataset

In [ ]:

full_dataset=Dataset_tox21(
    smiles=smiles,
    labels=labels,
    global_parameters=global_features,
    permitted_atoms=TYPE_ATOMS,
    hybrization_type=HYBRIDIZATION_TYPE,
    degree_atoms=ATOM_DEGREE,
    number_hydrogens=NUMBER_HYDROGENS,
    permitted_bonds=PERMITTED_BONDS,
    stereo_list=STEREO_LIST
)

dataset_size=len(full_dataset)
val_size = int(dataset_size * VAL_PERCENTAGE) 
test_size= int(dataset_size* TEST_PERCENTAGE)
train_size = dataset_size - val_size - test_size

print(f"Total Molecules: {dataset_size}")
print(f"Training set: {train_size}")
print(f"Test Set: {test_size}")

generator= torch.Generator().manual_seed(RANDOM_SEED)


train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, 
    [train_size, val_size, test_size], 
    generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)
print("Dataloaders ok")

for batch in train_loader:
    print(f"Batch: {batch}")
    print(f" - Batch x shape: {batch.x.shape}")    
    print(f" - Batch y shape: {batch.y.shape}")    
    print(f" - Batch edge_index: {batch.edge_index.shape}")
    break 


# **GNN Model**
--- 

In our model, node features are projected into a latent space of dim = `hidden_channels` through a linear layer, while the edge features are embedded with a `bond_encoder` composed of two linear layers, batch normalization and SiLU activation function.
Then, the model uses 5 sequential layers `GINEConv`. Each layer employs a two-layers MLP:
- Linear - BatchNorm - SiLU - Dropout
- Linear - BatchNorm -SiLU

In this way, each layer learns a representation each time more astract, progressively capturing local information from the graph. Then, node representations of each GINEConv layer is concatenated: this approach allows the model to preserve information coming from different depth level, avoiding oversmoothing of the information. Finally, a global add pooling and a global max pooling are applied on the concatenated representation, obtaining two vectors. These two vectors are then combined to keep general information (the sum) and predominant features (the max).
Eventually, the MLP is composed of two hidden layers with batch normalization, SiLU activation and Dropout. The final output is a vector of dimension `num_classes` (12 for Tox21) used for multi-class classification through `BCEWithLogitsLoss`.

In [ ]:
    
model = Tox21GNN(num_node_features=NODE_FEATURES,hidden_channels=HIDDEN_CHANNELS,num_classes=NUM_CLASSES,dropout=DROPOUT)
model = model.to(DEVICE)

#print(f"Using device: {DEVICE}")  
#print(model)        

# **Training**
---
1) As a first step, to handle the unbalance of classes, we compute weights for each class as:
\begin{equation}
\omega_i = \frac{N_{neg}}{N_{pos}}
\end{equation}
those weights are then used into the `BCEWithLogitsLoss`.

2) Training setup: Adam Optimizer with weight decay is used with a scheduler for the learning rate. We perform a linear warm-up on the 20% of total steps, then we compute a Cosine Annealing on the decay phase.

3) Training: each batch is moved on the GPU, then the forward pass with the GNN is computed and the masks on missing labels are applied with `apply_masks`. Finally, the loss is computed as 
\begin{equation}
\mathcal{L} = \frac{\sum_{mol,task}\mathcal{L}_{mol,task}}{\# \text{valid labels}}
\end{equation}
to take into account only valid labels. Then, backpropagation is carried out.



In [ ]:

print("--- Computing Class Weights ---")
class_weights = compute_class_weights(train_loader)
print(f"Computed weights: {class_weights}")


class_weights = class_weights.to(DEVICE)
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = int(total_steps * 0.20) 
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY) 
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights, reduction='none')
scheduler = create_lr_scheduler(optimizer, total_steps, warmup_steps)

#We have already trained and saved GNN_WEIGHTS IN MODEL_PATH
#print("--- Start Training ---")
#train(model, train_loader, val_loader,scheduler, criterion, optimizer)


# **Hyperparameter Search**
---
 
 Hyperparameter optimization is carried out using Optuna. The tuning process aims to maximize the validation ROC-AUC. For each trial, key hyperparameters—including hidden dimension, learning rate, weight decay, batch size, and dropout—are sampled and used to initialize a new Tox21GNN model. Models are trained using a class-weighted binary cross-entropy loss and a learning rate schedule with the same linear warm-up followed by cosine decay. Validation ROC-AUC is evaluated after each epoch and used both for optimization and early pruning. The best hyperparameter configuration is selected based on the highest validation performance.

In [ ]:

print("--- Hyperparameter Tuning con Optuna ---")

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)       
study = optuna.create_study(direction='maximize',sampler=sampler, pruner=optuna.pruners.MedianPruner())
study.optimize(
    lambda trial: objective(trial, class_weights, train_dataset, val_dataset),
    n_trials=60
)       

print("\n--- TUNING COMPLETED ---")
print("Best parameters:")
print(study.best_params)
print(f"Best ROC-AUC mean: {study.best_value}")

fig1 = plot_optimization_history(study)
fig1.show()
fig2 = plot_param_importances(study)
fig2.show()


best_param=study.best_params
print(best_param)

# **Test**
---
After training, the model parameters corresponding to the best validation performance are loaded and used to perform inference on the test set. The model outputs probability scores for each of the 12 Tox21 toxicity tests. Since the dataset contains missing labels, evaluation is carried out independently for each task by considering only the samples with valid annotations. For each test, the Receiver Operating Characteristic Area Under the Curve (ROC-AUC) is computed when both positive and negative classes are present; tasks containing only a single class are excluded from the evaluation. Additionally, predicted probabilities are thresholded at 0.5 to obtain binary predictions, which are used to compute the corresponding confusion matrices. Finally, an overall performance indicator is obtained by averaging the ROC-AUC scores across all evaluable tasks.

In [ ]:
print("--- Start Inference ---")

model.load_state_dict(torch.load(MODEL_PATH)) # upload weights
y_probs, y_true = get_predictions(model,DEVICE, test_loader) # inference

print(f"Inference Shape: {y_probs.shape}")
#print(y_probs)


task_names = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

roc_auc_scores = []
print("\n" + "="*50)
print(f"{'TASK NAME':<20} | {'ROC-AUC':<10} | {'Etichette Valide'}")
print("="*50)

list_confusion_matrix=[]
for i, name in enumerate(task_names):
    
    col_true = y_true[:, i]
    col_pred = y_probs[:, i]
    
    mask = ~np.isnan(col_true)
    valid_true = col_true[mask]
    valid_pred = col_pred[mask]
    
    if len(np.unique(valid_true)) < 2:
        print(f"{name:<20} | {'N/A':<10} | {len(valid_true)} (Solo una classe)")
        continue
        

    score = roc_auc_score(valid_true, valid_pred)
    roc_auc_scores.append(score)
    
    valid_pred = (valid_pred > 0.5).astype(int)
    valid_true = valid_true.astype(int)
    
    cm =confusion_matrix(valid_true, valid_pred, labels=[0, 1])
    list_confusion_matrix.append(cm)
    
    print(f"{name:<20} | {score:.4f}     | {len(valid_true)}")

print("-" * 50)
print(f"MEDIA TOTALE ROC-AUC: {np.mean(roc_auc_scores):.4f}")
print("=" * 50)

### ROC-AUC

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(task_names, roc_auc_scores, color='skyblue', edgecolor='navy')
plt.axhline(y=0.5, color='r', linestyle='--', label='Random Guess (0.5)')
plt.axhline(y=np.mean(roc_auc_scores), color='g', linestyle='-.', label=f'Mean: {np.mean(roc_auc_scores):.2f}')

plt.ylim(0.4, 1.0) 
plt.ylabel('ROC-AUC Score')
plt.title('Performance del Modello Tox21 GIN su ogni Task')
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)
for i,v in enumerate(roc_auc_scores):
    plt.text(i, v , f'{v:0.2f}%', color='black', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

### Confusion Matrix 

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()
for i, ax in enumerate(axes):
    task_name=task_names[i]
    
    sns.heatmap(list_confusion_matrix[i], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                annot_kws={"size": 14, "weight": "bold"})
    
    # Estetica
    ax.set_title(f"{task_name}", fontsize=14, fontweight='bold', color='navy')
    ax.set_xlabel('Predetto', fontsize=10)
    ax.set_ylabel('Reale', fontsize=10)
    ax.set_xticklabels(['Non-Toxic', 'Toxic'])
    ax.set_yticklabels(['Non-Toxic', 'Toxic'])